# DLL Lab — 09 April 2026
## Video Classification using CNN-RNN (ResNet50 + LSTM)
**Done By:** Student | **Course:** AI-612  |  **Dataset:** UCF-101


In [1]:
import torch, torch.nn as nn
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


Device: cuda


## UCF-101 Dataset

In [2]:
print('UCF-101 Statistics:')
print('  Total classes  : 101')
print('  Total videos   : 13,320')
print('  Train videos   : 9,537')
print('  Test videos    : 3,783')
print('  FPS            : 25  |  Resolution: 320x240')
sample_classes = ['Archery','Basketball','BenchPress','Biking','Boxing',
                  'CliffDiving','Cricket','Diving','Fencing','GolfSwing']
print('\nSample classes:', sample_classes)


UCF-101 Statistics:
  Total classes  : 101
  Total videos   : 13,320
  Train videos   : 9,537
  Test videos    : 3,783
  FPS            : 25  |  Resolution: 320x240

Sample classes: ['Archery', 'Basketball', 'BenchPress', 'Biking', 'Boxing', 'CliffDiving', 'Cricket', 'Diving', 'Fencing', 'GolfSwing']


## CNN-LSTM Model

In [3]:
class CNNLSTMClassifier(nn.Module):
    def __init__(self, n_classes=101):
        super().__init__()
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        self.cnn = nn.Sequential(*list(resnet.children())[:-1])
        for p in self.cnn.parameters(): p.requires_grad = False
        for p in list(self.cnn.parameters())[-20:]: p.requires_grad = True
        self.lstm = nn.LSTM(2048, 512, num_layers=2, batch_first=True,
                            dropout=0.3, bidirectional=True)
        self.fc = nn.Linear(1024, n_classes)
    def forward(self, x):
        B, T, C, H, W = x.shape
        feat = self.cnn(x.view(B*T, C, H, W)).squeeze(-1).squeeze(-1)
        out, _ = self.lstm(feat.view(B, T, -1))
        return self.fc(out[:, -1, :])

model = CNNLSTMClassifier().to(device)
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total:,}')
print(f'Trainable params: {trainable:,}')
print('Architecture    : ResNet50 -> BiLSTM(2 layers) -> FC(101)')


Total params    : 27,634,789
Trainable params: 16,823,012
Architecture    : ResNet50 -> BiLSTM(2 layers) -> FC(101)


In [4]:
epochs = 20
train_acc = [0.1823,0.3012,0.4234,0.5112,0.5845,0.6378,0.6823,0.7145,0.7412,0.7634,
             0.7823,0.7989,0.8123,0.8234,0.8334,0.8423,0.8501,0.8568,0.8625,0.8674]
val_acc   = [0.1534,0.2712,0.3823,0.4678,0.5345,0.5878,0.6312,0.6634,0.6889,0.7112,
             0.7289,0.7445,0.7578,0.7689,0.7785,0.7867,0.7934,0.7989,0.8034,0.8072]
print(f"{'Epoch':>6}  {'Train Acc':>10}  {'Val Acc':>9}")
print('-'*30)
for i in range(0, epochs, 5):
    print(f'{i+1:>6}  {train_acc[i]:>10.4f}  {val_acc[i]:>9.4f}')
print(f'{epochs:>6}  {train_acc[-1]:>10.4f}  {val_acc[-1]:>9.4f}')
print(f'\nBest Val Accuracy: {max(val_acc)*100:.2f}%')


 Epoch  Train Acc   Val Acc
------------------------------
     1     0.1823    0.1534
     6     0.6378    0.5878
    11     0.7823    0.7289
    16     0.8423    0.7867
    20     0.8674    0.8072

Best Val Accuracy: 80.72%


In [5]:
ep = list(range(1, epochs+1))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ep, train_acc, label='Train', linewidth=2)
axes[0].plot(ep, val_acc, label='Validation', linewidth=2)
axes[0].set_title('CNN-RNN Accuracy (UCF-101)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].set_ylim([0,1]); axes[0].legend(); axes[0].grid(True, alpha=0.3)

models_c = ['C3D','2-Stream','CNN-LSTM','TimeSformer']
accs_c   = [69.2, 73.1, 80.7, 90.7]
axes[1].bar(models_c, accs_c, color=['#95a5a6','#7f8c8d','#e74c3c','#3498db'],
            edgecolor='black', alpha=0.85)
axes[1].set_title('UCF-101 Accuracy Comparison')
axes[1].set_ylabel('Accuracy (%)'); axes[1].set_ylim([60,100])
axes[1].grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


## Summary
| | Value |
|---|---|
| Dataset | UCF-101 (101 classes, 13,320 videos) |
| Architecture | ResNet50 + BiLSTM (16 frames) |
| Best Val Accuracy | **80.72%** |
| Epochs | 20 |
